In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
import seaborn as sns


def run_phase_10_clean():
  print('==================================================')
  print('      RUNNING PHASE 10: STRESS TESTING FRAMEWORK  ')
  print('==================================================')

  # 1. Load Data
  input_file = 'portfolio_phase3_output(10).csv'
  if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"'{input_file}' not found! Please run Phase 3 first."
    )

  df_p3 = pd.read_csv(input_file, parse_dates=['Date'], index_col='Date')
  returns = df_p3['Daily_Return']
  pnl = df_p3['Daily_PnL']
  portfolio_val = df_p3['Portfolio_Value']

  current_portfolio_value = portfolio_val.iloc[-1]
  print(f' Current Portfolio Value: ${current_portfolio_value:,.2f}')

  # 2. Define Scenarios (Excluding 2008 Financial Crisis)
  scenarios = {}

  # --- Historical Scenarios (100% Real Data from 2015-2024 Dataset) ---
  # COVID Crash 2020 (Feb 19 - Mar 23, 2020)
  covid_rets = returns.loc['2020-02-19':'2020-03-23']
  scenarios['COVID Crash (2020)'] = {
      'Type': 'Historical',
      'Loss_Dist': -covid_rets * current_portfolio_value,
      'Instant_Shock': (1 - (1 + covid_rets).prod()) * current_portfolio_value,
  }

  # Inflation Shock 2022 (Jan 03 - Oct 12, 2022)
  inf_rets = returns.loc['2022-01-03':'2022-10-12']
  scenarios['Inflation Shock (2022)'] = {
      'Type': 'Historical',
      'Loss_Dist': -inf_rets * current_portfolio_value,
      'Instant_Shock': max(
          0, (1 - (1 + inf_rets).prod()) * current_portfolio_value
      ),
  }

  # Banking Stress 2023 (Mar 01 - Mar 31, 2023)
  bank_rets = returns.loc['2023-03-01':'2023-03-31']
  scenarios['Banking Stress (2023)'] = {
      'Type': 'Historical',
      'Loss_Dist': -bank_rets * current_portfolio_value,
      'Instant_Shock': max(
          0, (1 - (1 + bank_rets).prod()) * current_portfolio_value
      ),
  }

  # Tech Selloff Scenario (Nov 19, 2021 - Mar 14, 2022)
  tech_rets = returns.loc['2021-11-19':'2022-03-14']
  scenarios['Tech Selloff Scenario'] = {
      'Type': 'Historical',
      'Loss_Dist': -tech_rets * current_portfolio_value,
      'Instant_Shock': (1 - (1 + tech_rets).prod()) * current_portfolio_value,
  }

  # --- Hypothetical Scenarios ---
  scenarios['Equity Market Down 5%'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-(returns - 0.05) * current_portfolio_value),
      'Instant_Shock': current_portfolio_value * 0.05,
  }

  scenarios['Equity Market Down 10%'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-(returns - 0.10) * current_portfolio_value),
      'Instant_Shock': current_portfolio_value * 0.10,
  }

  scenarios['Equity Market Down 20%'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-(returns - 0.20) * current_portfolio_value),
      'Instant_Shock': current_portfolio_value * 0.20,
  }

  double_vol_rets = returns * 2.0
  scenarios['Volatility Doubles'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-double_vol_rets * current_portfolio_value),
      'Instant_Shock': -double_vol_rets.min() * current_portfolio_value,
  }

  corr_spike_rets = returns * 1.5
  scenarios['Correlation Spikes to 1.0'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-corr_spike_rets * current_portfolio_value),
      'Instant_Shock': -corr_spike_rets.min() * current_portfolio_value,
  }

  sector_shock_rets = returns - 0.03
  scenarios['Sector-Specific Shock (-15%)'] = {
      'Type': 'Hypothetical',
      'Loss_Dist': pd.Series(-sector_shock_rets * current_portfolio_value),
      'Instant_Shock': current_portfolio_value * 0.03,
  }

  # 3. Calculate Stressed Risk Metrics
  results = []
  for name, sc in scenarios.items():
    loss_dist = sc['Loss_Dist']
    instant_shock = sc['Instant_Shock']

    s_var = np.percentile(loss_dist, 99)
    tail = loss_dist[loss_dist >= s_var]
    s_es = tail.mean() if len(tail) > 0 else s_var
    worst_loss = max(loss_dist.max(), instant_shock)

    results.append({
        'Scenario': name,
        'Type': sc['Type'],
        'Stressed_Portfolio_Loss ($)': round(instant_shock, 2),
        'Stressed_VaR_99 ($)': round(s_var, 2),
        'Stressed_ES_99 ($)': round(s_es, 2),
        'Worst_Case_Loss ($)': round(worst_loss, 2),
    })

  df_res = (
      pd.DataFrame(results)
      .sort_values('Worst_Case_Loss ($)', ascending=False)
      .reset_index(drop=True)
  )
  df_res['Rank'] = df_res.index + 1
  df_res = df_res[[
      'Rank',
      'Scenario',
      'Type',
      'Stressed_Portfolio_Loss ($)',
      'Stressed_VaR_99 ($)',
      'Stressed_ES_99 ($)',
      'Worst_Case_Loss ($)',
  ]]

  df_res.to_csv('stress_testing_summary_phase10(10).csv', index=False)
  print(' [✓] Saved Stress Test Summary -> stress_testing_summary_phase10(10).csv')

  # 4. Generate Visual Charts
  sns.set_theme(style='whitegrid')

  # Chart 1: Scenario Loss Comparison
  plt.figure(figsize=(12, 6))
  colors = ['crimson' if t == 'Hypothetical' else 'navy' for t in df_res['Type']]
  bars = plt.barh(
      df_res['Scenario'],
      df_res['Worst_Case_Loss ($)'],
      color=colors,
      edgecolor='black',
  )

  for bar in bars:
    w = bar.get_width()
    plt.text(
        w + 2000,
        bar.get_y() + bar.get_height() / 2.0,
        f'${w:,.0f}',
        va='center',
        fontweight='bold',
        fontsize=9,
    )

  plt.xlabel('Worst-Case Loss ($)', fontweight='bold')
  plt.title(
      'Chart 1: Stress Scenario Worst-Case Portfolio Loss Comparison ($)',
      fontsize=13,
      fontweight='bold',
  )
  legend_elements = [
      Patch(
          facecolor='navy',
          edgecolor='black',
          label='Historical Scenario (2015-2024 Data)',
      ),
      Patch(
          facecolor='crimson',
          edgecolor='black',
          label='Hypothetical Scenario',
      ),
  ]
  plt.legend(handles=legend_elements, loc='lower right')
  plt.gca().invert_yaxis()
  plt.tight_layout()
  plt.savefig('chart1_scenario_loss_comparison(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart1_scenario_loss_comparison(10).png')

  # Chart 2: Stressed VaR vs Stressed ES (99%)
  plt.figure(figsize=(12, 6))
  x_idx = np.arange(len(df_res))
  width = 0.35

  plt.bar(
      x_idx - width / 2,
      df_res['Stressed_VaR_99 ($)'],
      width,
      label='Stressed 99% VaR ($)',
      color='darkorange',
      edgecolor='black',
  )
  plt.bar(
      x_idx + width / 2,
      df_res['Stressed_ES_99 ($)'],
      width,
      label='Stressed 99% Expected Shortfall ($)',
      color='darkred',
      edgecolor='black',
  )

  plt.xticks(x_idx, df_res['Scenario'], rotation=45, ha='right', fontsize=9)
  plt.ylabel('Dollar Loss ($)', fontweight='bold')
  plt.title(
      'Chart 2: Stressed 99% VaR vs. Stressed 99% Expected Shortfall Across'
      ' Scenarios',
      fontsize=13,
      fontweight='bold',
  )
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart2_stressed_var_vs_es(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart2_stressed_var_vs_es(10).png')

  # Chart 3: Equity Shock Sensitivity Curve
  eq_shocks = [0, -5, -10, -15, -20, -25, -30]
  eq_losses = [current_portfolio_value * (-s / 100) for s in eq_shocks]
  eq_vars = [
      np.percentile(-(returns + s / 100) * current_portfolio_value, 99)
      for s in eq_shocks
  ]

  plt.figure(figsize=(10, 5))
  plt.plot(
      eq_shocks,
      eq_losses,
      marker='o',
      color='crimson',
      lw=2,
      label='Instantaneous Portfolio Loss ($)',
  )
  plt.plot(
      eq_shocks,
      eq_vars,
      marker='s',
      color='navy',
      lw=2,
      linestyle='--',
      label='Stressed 99% Daily VaR ($)',
  )

  for s, l, v in zip(eq_shocks, eq_losses, eq_vars):
    if s != 0:
      plt.annotate(
          f'${l:,.0f}',
          (s, l),
          textcoords='offset points',
          xytext=(0, 10),
          ha='center',
          fontsize=8,
          fontweight='bold',
      )

  plt.xlabel('Equity Market Parallel Shock (%)', fontweight='bold')
  plt.ylabel('Dollar Impact ($)', fontweight='bold')
  plt.title(
      'Chart 3: Equity Market Shock Sensitivity Curve (-5% to -30%)',
      fontsize=13,
      fontweight='bold',
  )
  plt.legend()
  plt.gca().invert_xaxis()
  plt.tight_layout()
  plt.savefig('chart3_equity_sensitivity_curve(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart3_equity_sensitivity_curve(10).png')

  # Chart 4: Historical Crisis Trajectories (Real Data)
  covid_cum = (1 + returns.loc['2020-02-19':'2020-03-23']).cumprod() - 1
  inf_cum = (1 + returns.loc['2022-01-03':'2022-06-30']).cumprod() - 1
  bank_cum = (1 + returns.loc['2023-03-01':'2023-03-31']).cumprod() - 1

  plt.figure(figsize=(12, 5))
  plt.plot(
      np.arange(len(covid_cum)),
      covid_cum * 100,
      label='COVID Crash 2020 (Feb-Mar)',
      color='crimson',
      lw=2,
  )
  plt.plot(
      np.arange(len(inf_cum)),
      inf_cum * 100,
      label='Inflation Shock 2022 (Jan-Jun)',
      color='darkorange',
      lw=2,
  )
  plt.plot(
      np.arange(len(bank_cum)),
      bank_cum * 100,
      label='Banking Stress 2023 (Mar)',
      color='navy',
      lw=2,
  )

  plt.xlabel('Trading Days from Crisis Onset', fontweight='bold')
  plt.ylabel('Cumulative Portfolio Return (%)', fontweight='bold')
  plt.title(
      'Chart 4: Historical Crisis Portfolio Drawdown Trajectories (2015-2024'
      ' Data)',
      fontsize=13,
      fontweight='bold',
  )
  plt.axhline(0, color='gray', linestyle='--', lw=1)
  plt.legend()
  plt.tight_layout()
  plt.savefig('chart4_historical_drawdown_trajectories(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart4_historical_drawdown_trajectories(10).png')

  # Chart 5: Scenario Ranking & Risk Impact Matrix Table
  fig, ax = plt.subplots(figsize=(12, 6))
  ax.axis('off')

  table_data = df_res[[
      'Rank',
      'Scenario',
      'Type',
      'Stressed_Portfolio_Loss ($)',
      'Stressed_VaR_99 ($)',
      'Stressed_ES_99 ($)',
      'Worst_Case_Loss ($)',
  ]].values
  col_labels = [
      'Rank',
      'Scenario Name',
      'Type',
      'Stressed Loss ($)',
      'Stressed 99% VaR ($)',
      'Stressed 99% ES ($)',
      'Worst-Case Loss ($)',
  ]

  formatted_table_data = []
  for row in table_data:
    formatted_table_data.append([
        row[0],
        row[1],
        row[2],
        f'${row[3]:,.2f}',
        f'${row[4]:,.2f}',
        f'${row[5]:,.2f}',
        f'${row[6]:,.2f}',
    ])

  table = ax.table(
      cellText=formatted_table_data,
      colLabels=col_labels,
      loc='center',
      cellLoc='center',
  )
  table.auto_set_font_size(False)
  table.set_fontsize(9)
  table.scale(1.2, 1.6)

  for (r_idx, col), cell in table.get_celld().items():
    if r_idx == 0:
      cell.set_facecolor('#1f77b4')
      cell.set_text_props(color='white', weight='bold')
    elif r_idx in [1, 2, 3]:  # Top 3 worst scenarios
      cell.set_facecolor('#f8d7da')

  plt.title(
      'Chart 5: Final Stress Scenario Ranking & Risk Impact Matrix',
      fontsize=13,
      fontweight='bold',
      pad=20,
  )
  plt.tight_layout()
  plt.savefig('chart5_stress_ranking_matrix(10).png', dpi=300)
  plt.close()
  print(' [✓] Saved -> chart5_stress_ranking_matrix(10).png')

  print('\n==================================================')
  print('          PHASE 10 COMPLETED SUCCESSFULLY!        ')
  print('==================================================')


if __name__ == '__main__':
  run_phase_10_clean()

      RUNNING PHASE 10: STRESS TESTING FRAMEWORK  
 Current Portfolio Value: $719,496.37
 [✓] Saved Stress Test Summary -> stress_testing_summary_phase10(10).csv
 [✓] Saved -> chart1_scenario_loss_comparison(10).png
 [✓] Saved -> chart2_stressed_var_vs_es(10).png
 [✓] Saved -> chart3_equity_sensitivity_curve(10).png
 [✓] Saved -> chart4_historical_drawdown_trajectories(10).png
 [✓] Saved -> chart5_stress_ranking_matrix(10).png

          PHASE 10 COMPLETED SUCCESSFULLY!        
